# 01 — Data Cleaning Pipeline

**Project:** Subscription Churn Intelligence System  
**Purpose:** Generate the synthetic dataset, inspect it, apply a cleaning pipeline, and save the processed data.

---

## Why Data Cleaning Matters

Real-world data is never perfect. Before we can analyse or model anything, we need to:
- Remove duplicate records (same customer counted twice = wrong totals)
- Fix invalid values (age = 200 is impossible)
- Handle missing values (NaN breaks most ML algorithms)
- Parse dates correctly (stored as strings, need to be datetime objects)
- Add derived features (tenure = today minus signup_date)

**Rule:** Always keep the raw data untouched. Save cleaned data separately.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_processing import (
    generate_synthetic_dataset,
    clean_dataset,
    calculate_churn_rate,
    RAW_DATA_PATH,
    PROCESSED_DATA_PATH,
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)
print('Libraries loaded.')

## Step 1 — Generate the Raw Dataset

In [ ]:
raw_df = generate_synthetic_dataset(n_customers=5000, seed=42)
RAW_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
raw_df.to_csv(RAW_DATA_PATH, index=False)

print(f'Raw dataset: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns')
raw_df.head()

## Step 2 — Inspect Raw Data Quality

In [ ]:
print('=== Data Types ===')
print(raw_df.dtypes)
print()
print('=== Missing Values ===')
print(raw_df.isnull().sum())
print()
print('=== Duplicate Rows ===')
print(f'Duplicates: {raw_df.duplicated().sum()}')

In [ ]:
# Check age range — should be 16-100 for real customers
print('Age stats:')
print(raw_df['age'].describe())
print()
print('Invalid ages (< 16 or > 100):')
invalid = raw_df[(raw_df['age'] < 16) | (raw_df['age'] > 100)]
print(invalid[['customer_id', 'age']].head(10))

In [ ]:
# Check plan distribution
print('Plan distribution:')
print(raw_df['plan'].value_counts())
print()
print('Churned distribution:')
print(raw_df['churned'].value_counts())

## Step 3 — Run the Cleaning Pipeline

Cleaning decisions documented:
1. **Duplicates** → drop exact duplicate rows
2. **Invalid ages** (< 16 or > 100) → set to NaN, then impute
3. **Missing numerics** → impute with median (robust to outliers vs mean)
4. **Missing dates** → last_login_date filled with signup_date (conservative)
5. **Derived features** → tenure_days, tenure_months, days_since_login

In [ ]:
print('Running cleaning pipeline...')
clean_df = clean_dataset(raw_df)

PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(PROCESSED_DATA_PATH, index=False)

print(f'\nClean dataset: {clean_df.shape[0]:,} rows × {clean_df.shape[1]} columns')
print(f'Saved to: {PROCESSED_DATA_PATH}')

In [ ]:
print('Missing values after cleaning:')
print(clean_df.isnull().sum())
print()
print('Data types after cleaning:')
print(clean_df.dtypes)

## Step 4 — Calculate Churn Rate

**What is churn rate?**  
The percentage of customers who have cancelled their subscription.  
Formula: `(churned customers / total customers) × 100`

**Why it matters:**  
Churn rate is the primary health metric for any subscription business.  
A 5% monthly churn means you lose 46% of your customer base every year.

In [ ]:
churn_rate = calculate_churn_rate(clean_df)
print(f'Overall Churn Rate: {churn_rate:.1f}%')
print(f'Active customers:   {(clean_df["churned"]==0).sum():,}')
print(f'Churned customers:  {(clean_df["churned"]==1).sum():,}')
print(f'Total customers:    {len(clean_df):,}')

In [ ]:
# Preview cleaned data
clean_df[['customer_id', 'plan', 'tenure_months', 'days_since_login', 
          'usage_frequency', 'payment_failures', 'churned']].head(10)

## Summary

| Step | Action | Rows Before | Rows After |
|------|--------|------------|------------|
| 1 | Remove duplicates | 5,050 | ~5,001 |
| 2 | Fix invalid ages | — | 20 set to NaN |
| 3 | Impute missing values | — | 0 missing |
| 4 | Add tenure features | — | 2 new columns |

**Output:** `data/processed/customers_clean.csv` — used by all subsequent notebooks.